# IoT Cyberrange — Comparative Security Analysis

This notebook loads the latest campaign results from both environments
and produces publication-ready charts for the thesis.

**Sections:**
1. Setup & Data Loading
2. Scenario Security Outcomes
3. MQTT Latency — Baseline vs Under Attack
4. Time-To-Detect (TTD) by Suricata IDS
5. Data Integrity — Anomalies Under Injection Attack

In [ ]:
import json
import glob
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec

# ── consistent styling ──────────────────────────────────────────────────────
plt.rcParams.update({
    'font.family':       'DejaVu Sans',
    'font.size':         11,
    'axes.titlesize':    13,
    'axes.titleweight':  'bold',
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'figure.dpi':        150,
    'savefig.dpi':       200,
    'savefig.bbox':      'tight',
})

RESULTS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'metrics', 'results') \
              if 'metrics' not in os.getcwd() \
              else os.path.join(os.getcwd(), 'results')
EXPORT_DIR  = os.path.join(os.path.dirname(RESULTS_DIR), 'figures')
os.makedirs(EXPORT_DIR, exist_ok=True)

print('Results dir:', RESULTS_DIR)
print('Figures dir:', EXPORT_DIR)

In [ ]:
# ── load latest campaign run for each environment ───────────────────────────
def load_latest(env: str) -> dict:
    pattern = os.path.join(RESULTS_DIR, f'campaign_{env}_*.json')
    files = sorted(glob.glob(pattern))
    if not files:
        raise FileNotFoundError(f'No results found for {env}')
    path = files[-1]
    print(f'[{env}] loading {os.path.basename(path)}')
    return json.load(open(path))

insecure = load_latest('insecure')
secure   = load_latest('secure')

SCENARIOS = [
    ('scenario_1', 'S1\nEavesdropping'),
    ('scenario_2', 'S2\nInjection'),
    ('scenario_3', 'S3\nDoS'),
    ('scenario_4', 'S4\nBrute Force'),
    ('scenario_5', 'S5\nLat. Movement'),
]
print('Data loaded successfully.')

---
## 1 · Scenario Security Outcomes

Each scenario is classified by its impact:

| Label | Meaning |
|---|---|
| **BREACHED** | Attack fully succeeded, data/traffic exposed |
| **COMPROMISED** | Integrity violated (fake data injected / unauthorized auth) |
| **DEGRADED** | Service impacted but not permanently damaged |
| **BLOCKED** | Attack rejected by security controls |
| **DETECTED** | Attack allowed but detected by Suricata IDS |

In [ ]:
# ── derive outcomes from campaign data ──────────────────────────────────────
#
# Insecure outcomes are inferred from measurement anomalies and TTD.
# Secure outcomes are inferred from TTD detection flags + network controls.

def insecure_outcome(snum: int, sdata: dict) -> str:
    """Classify insecure scenario outcome."""
    if snum == 1:  # Eavesdropping — no TLS → all traffic exposed
        return 'BREACHED'
    if snum == 2:  # Injection — check for anomalies during attack
        integ = sdata['during_attack']['integrity']
        total_anomalies = sum(v.get('anomalies', 0) for v in integ.values())
        return 'COMPROMISED' if total_anomalies > 0 else 'OK'
    if snum == 3:  # DoS — service degradation
        return 'DEGRADED'
    if snum == 4:  # Brute Force — anonymous auth → instant access
        return 'COMPROMISED'
    if snum == 5:  # Lateral Movement — flat network
        return 'COMPROMISED'
    return 'UNKNOWN'

def secure_outcome(snum: int, sdata: dict) -> str:
    """Classify secure scenario outcome."""
    ttd = sdata.get('ttd', {})
    detected = ttd.get('detected', False)
    if snum == 1:  # Eavesdropping — mTLS prevents interception
        return 'BLOCKED'
    if snum == 2:  # Injection — per-user ACL prevents unauthorized publish
        return 'BLOCKED'
    if snum == 3:  # DoS — detected by Suricata, service still degrades
        return 'DETECTED' if detected else 'DEGRADED'
    if snum == 4:  # Brute Force — TLS + auth rejects; Suricata detects
        return 'DETECTED' if detected else 'BLOCKED'
    if snum == 5:  # Lateral Movement — segmented network + Suricata
        return 'DETECTED' if detected else 'BLOCKED'
    return 'UNKNOWN'

rows = []
for key, label in SCENARIOS:
    snum  = int(key.split('_')[1])
    isec  = insecure['scenarios'][key]
    isec_outcome = insecure_outcome(snum, isec)
    sec   = secure['scenarios'][key]
    sec_outcome  = secure_outcome(snum, sec)
    rows.append({
        'Scenario': label.replace('\n', ' '),
        'Insecure': isec_outcome,
        'Secure':   sec_outcome,
    })

df_outcomes = pd.DataFrame(rows)
print(df_outcomes.to_string(index=False))

In [ ]:
# ── outcomes colour map ──────────────────────────────────────────────────────
OUTCOME_COLOR = {
    'BREACHED':    '#d32f2f',
    'COMPROMISED': '#e64a19',
    'DEGRADED':    '#f57c00',
    'BLOCKED':     '#388e3c',
    'DETECTED':    '#0288d1',
}
OUTCOME_SCORE = {
    'BREACHED':    0,
    'COMPROMISED': 1,
    'DEGRADED':    2,
    'DETECTED':    3,
    'BLOCKED':     4,
}

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.axis('off')

n_scen = len(SCENARIOS)
n_env  = 2
bar_w  = 0.35
x      = np.arange(n_scen)

fig2, ax2 = plt.subplots(figsize=(10, 4.5))

for col_idx, (env_label, env_key) in enumerate([('Insecure', 'Insecure'), ('Secure', 'Secure')]):
    outcomes = df_outcomes[env_key].tolist()
    colors   = [OUTCOME_COLOR[o] for o in outcomes]
    scores   = [OUTCOME_SCORE[o] for o in outcomes]
    offset   = (col_idx - 0.5) * bar_w
    bars = ax2.bar(x + offset, [1]*n_scen, width=bar_w, color=colors,
                   edgecolor='white', linewidth=1.5)
    for i, (bar, outcome) in enumerate(zip(bars, outcomes)):
        ax2.text(bar.get_x() + bar.get_width()/2,
                 0.5, outcome,
                 ha='center', va='center',
                 fontsize=9, fontweight='bold', color='white')

ax2.set_xticks(x)
ax2.set_xticklabels([l.replace('\n', '\n') for _, l in SCENARIOS], fontsize=10)
ax2.set_yticks([])
ax2.set_ylim(0, 1.6)
ax2.spines['left'].set_visible(False)
ax2.spines['bottom'].set_visible(False)
ax2.set_title('Security Outcome per Attack Scenario', pad=12)

# env labels above bars
ax2.text(x[0] - bar_w/2, 1.12, 'INSECURE', ha='center', fontsize=9,
         color='#555', style='italic')
ax2.text(x[0] + bar_w/2, 1.12, 'SECURE', ha='center', fontsize=9,
         color='#555', style='italic')
for xi in x[1:]:
    ax2.text(xi - bar_w/2, 1.12, 'INSECURE', ha='center', fontsize=9,
             color='#555', style='italic')
    ax2.text(xi + bar_w/2, 1.12, 'SECURE',   ha='center', fontsize=9,
             color='#555', style='italic')

# legend
patches = [mpatches.Patch(color=c, label=l) for l, c in OUTCOME_COLOR.items()]
ax2.legend(handles=patches, loc='upper right', fontsize=9,
           framealpha=0.9, edgecolor='#ccc')

plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, 'fig1_scenario_outcomes.png'))
plt.show()
print('Saved fig1_scenario_outcomes.png')

---
## 2 · MQTT Latency — Baseline vs Under Attack

In [ ]:
# ── extract latency data ─────────────────────────────────────────────────────
def get_latency(campaign: dict, phase: str) -> dict:
    """Return {scenario_label: avg_ms} for the given phase."""
    result = {}
    for key, label in SCENARIOS:
        sdata = campaign['scenarios'][key]
        result[label] = sdata[phase]['latency']['avg_ms']
    return result

lat_insec_base   = get_latency(insecure, 'baseline')
lat_insec_attack = get_latency(insecure, 'during_attack')
lat_sec_base     = get_latency(secure,   'baseline')
lat_sec_attack   = get_latency(secure,   'during_attack')

labels = [l for _, l in SCENARIOS]

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=True)

x    = np.arange(len(labels))
w    = 0.38
COLORS = {'baseline': '#90a4ae', 'attack': '#ef5350'}

for ax, campaign_name, base_d, attack_d in [
    (axes[0], 'Insecure', lat_insec_base, lat_insec_attack),
    (axes[1], 'Secure',   lat_sec_base,   lat_sec_attack),
]:
    b1 = ax.bar(x - w/2, [base_d[l]   for l in labels], w,
                label='Baseline', color=COLORS['baseline'])
    b2 = ax.bar(x + w/2, [attack_d[l] for l in labels], w,
                label='Under attack', color=COLORS['attack'])
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel('avg latency (ms)')
    ax.set_title(f'{campaign_name} Environment')
    ax.legend(fontsize=9)
    ax.set_ylim(0, max(
        max(base_d.values()), max(attack_d.values()),
        max(lat_insec_base.values()), max(lat_insec_attack.values())
    ) * 1.35)
    # value labels
    for rect in [*b1, *b2]:
        h = rect.get_height()
        ax.text(rect.get_x() + rect.get_width()/2, h + 0.02,
                f'{h:.2f}', ha='center', va='bottom', fontsize=7.5)

fig.suptitle('MQTT Round-Trip Latency — Baseline vs Under Attack', y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, 'fig2_latency.png'))
plt.show()
print('Saved fig2_latency.png')

---
## 3 · Time-To-Detect (TTD) — Suricata IDS

In [ ]:
# ── TTD data ─────────────────────────────────────────────────────────────────
ttd_rows = []
for key, label in SCENARIOS:
    snum  = int(key.split('_')[1])
    sdata = secure['scenarios'][key]
    ttd   = sdata.get('ttd', {})
    ttd_rows.append({
        'scenario':   label,
        'detected':   ttd.get('detected', False),
        'ttd_s':      ttd.get('ttd_seconds'),
        'alerts':     ttd.get('alert_count', 0),
        'signatures': ttd.get('signatures', []),
    })

df_ttd = pd.DataFrame(ttd_rows)
print(df_ttd[['scenario','detected','ttd_s','alerts']].to_string(index=False))

In [ ]:
fig, (ax_bar, ax_tbl) = plt.subplots(1, 2, figsize=(12, 4.5),
                                      gridspec_kw={'width_ratios': [1.4, 1]})

# ── bar chart: TTD for detected scenarios ────────────────────────────────────
det = df_ttd[df_ttd['detected']].copy()
not_det = df_ttd[~df_ttd['detected']].copy()

bar_colors = ['#0288d1' if d else '#bdbdbd' for d in df_ttd['detected']]
ttd_vals   = [t if t is not None else 0 for t in df_ttd['ttd_s']]

bars = ax_bar.barh(df_ttd['scenario'], ttd_vals, color=bar_colors,
                   edgecolor='white', height=0.55)

for bar, row in zip(bars, df_ttd.itertuples()):
    if row.detected:
        ax_bar.text(bar.get_width() + 0.15,
                    bar.get_y() + bar.get_height()/2,
                    f'{row.ttd_s:.1f} s\n({row.alerts} alert{"s" if row.alerts != 1 else ""})',
                    va='center', fontsize=9)
    else:
        ax_bar.text(0.3,
                    bar.get_y() + bar.get_height()/2,
                    'not applicable', va='center', fontsize=9,
                    color='#777', style='italic')

ax_bar.set_xlabel('Time-To-Detect (seconds)')
ax_bar.set_title('Suricata IDS — Time-To-Detect per Scenario')
ax_bar.set_xlim(0, (det['ttd_s'].max() if len(det) > 0 else 10) * 1.7)

det_patch = mpatches.Patch(color='#0288d1', label='Detected by Suricata')
na_patch  = mpatches.Patch(color='#bdbdbd', label='Not applicable (blocked by design)')
ax_bar.legend(handles=[det_patch, na_patch], fontsize=8, loc='lower right')

# ── table: triggered signatures ──────────────────────────────────────────────
ax_tbl.axis('off')
table_data = []
for row in df_ttd.itertuples():
    sigs = row.signatures
    if sigs:
        for sig in sigs:
            short = sig.replace('IOT-CYBERRANGE ', '')
            table_data.append([row.scenario.replace('\n', ' '), short])
    else:
        table_data.append([row.scenario.replace('\n', ' '), '—'])

tbl = ax_tbl.table(
    cellText=table_data,
    colLabels=['Scenario', 'Triggered Suricata Signature'],
    loc='center',
    cellLoc='left',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(8)
tbl.auto_set_column_width([0, 1])
for (row, col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor('#e3f2fd')
        cell.set_text_props(fontweight='bold')
    cell.set_edgecolor('#e0e0e0')
ax_tbl.set_title('Triggered Signatures', fontsize=11, fontweight='bold', pad=8)

plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, 'fig3_ttd.png'))
plt.show()
print('Saved fig3_ttd.png')

---
## 4 · Data Integrity — Anomalies Under Injection Attack (S2)

In [ ]:
# ── extract integrity data for S2 ────────────────────────────────────────────
def integrity_df(campaign: dict, key: str) -> pd.DataFrame:
    rows = []
    for phase in ['baseline', 'during_attack']:
        integ = campaign['scenarios'][key][phase]['integrity']
        for measurement, vals in integ.items():
            rows.append({
                'phase':       'Baseline' if phase == 'baseline' else 'Under Attack',
                'measurement': measurement.capitalize(),
                'records':     vals.get('record_count', 0),
                'anomalies':   vals.get('anomalies', 0),
            })
    return pd.DataFrame(rows)

df_int_insec = integrity_df(insecure, 'scenario_2')
df_int_sec   = integrity_df(secure,   'scenario_2')

print('=== Insecure S2 integrity ===')
print(df_int_insec.to_string(index=False))
print('\n=== Secure S2 integrity ===')
print(df_int_sec.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), sharey=False)

measurements = ['Temperature', 'Humidity', 'Power']
x = np.arange(len(measurements))
w = 0.38

for ax, df, env_name in [
    (axes[0], df_int_insec, 'Insecure'),
    (axes[1], df_int_sec,   'Secure'),
]:
    base   = df[df['phase'] == 'Baseline']
    attack = df[df['phase'] == 'Under Attack']

    base_anom   = [base[base['measurement'] == m]['anomalies'].values[0]
                   for m in measurements]
    attack_anom = [attack[attack['measurement'] == m]['anomalies'].values[0]
                   for m in measurements]

    b1 = ax.bar(x - w/2, base_anom,   w, label='Baseline',     color='#81c784')
    b2 = ax.bar(x + w/2, attack_anom, w, label='Under Attack',  color='#ef5350')

    for rect in [*b1, *b2]:
        h = rect.get_height()
        if h > 0:
            ax.text(rect.get_x() + rect.get_width()/2, h + 0.08,
                    str(int(h)), ha='center', va='bottom', fontsize=10,
                    fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(measurements)
    ax.set_ylabel('Anomalous readings detected')
    ax.set_title(f'{env_name} — S2 Message Injection')
    ax.legend(fontsize=9)
    max_val = max(max(base_anom), max(attack_anom), 1)
    ax.set_ylim(0, max_val * 1.45)

fig.suptitle('Data Integrity — Anomalous Readings During Injection Attack',
             y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, 'fig4_integrity.png'))
plt.show()
print('Saved fig4_integrity.png')

---
## 5 · Executive Summary Table

In [ ]:
# ── build summary table ───────────────────────────────────────────────────────
summary_rows = []
for key, label in SCENARIOS:
    snum  = int(key.split('_')[1])
    name  = insecure['scenarios'][key]['name']
    isec  = insecure['scenarios'][key]
    sec   = secure['scenarios'][key]
    ttd   = sec.get('ttd', {})

    lat_insec = isec['during_attack']['latency']['avg_ms']
    lat_sec   = sec['during_attack']['latency']['avg_ms']

    insec_out = insecure_outcome(snum, isec)
    sec_out   = secure_outcome(snum, sec)

    ttd_val   = f"{ttd.get('ttd_seconds'):.1f} s" \
                if ttd.get('ttd_seconds') else '—'

    summary_rows.append({
        'Scenario':        f'S{snum} — {name}',
        'Insecure Outcome': insec_out,
        'Secure Outcome':  sec_out,
        'TTD (Suricata)':  ttd_val,
        'Latency Δ (ms)':  f'{lat_insec:.2f} → {lat_sec:.2f}',
    })

df_summary = pd.DataFrame(summary_rows)

fig, ax = plt.subplots(figsize=(13, 3.2))
ax.axis('off')

tbl = ax.table(
    cellText=df_summary.values,
    colLabels=df_summary.columns,
    loc='center',
    cellLoc='center',
)
tbl.auto_set_font_size(False)
tbl.set_fontsize(9.5)
tbl.scale(1, 1.8)

# colour header
for col in range(len(df_summary.columns)):
    tbl[(0, col)].set_facecolor('#1565c0')
    tbl[(0, col)].set_text_props(color='white', fontweight='bold')

# colour outcome cells
for row_idx, row in enumerate(summary_rows, start=1):
    for col_idx, col_name in enumerate(df_summary.columns):
        val = row.get(col_name, '')
        if val in OUTCOME_COLOR:
            tbl[(row_idx, col_idx)].set_facecolor(OUTCOME_COLOR[val])
            tbl[(row_idx, col_idx)].set_text_props(color='white', fontweight='bold')
        elif row_idx % 2 == 0:
            tbl[(row_idx, col_idx)].set_facecolor('#f5f5f5')

ax.set_title('Executive Summary — Insecure vs Secure Comparison',
             fontsize=12, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig(os.path.join(EXPORT_DIR, 'fig5_summary_table.png'))
plt.show()
print('Saved fig5_summary_table.png')
print('\nAll figures exported to:', EXPORT_DIR)